# Project Nord SNN + crdt-merge Integration
Run on A100. Install deps first.

In [ ]:
!pip install -q crdt-merge numpy

## 1. Federated Training — Merge checkpoints from multiple GPUs

In [ ]:
import numpy as np
from crdt_merge.model.federated import FederatedMerge
from crdt_merge.model.crdt_state import CRDTMergeState
from crdt_merge.core import GCounter, PNCounter, LWWRegister, LWWMap, ORSet
from crdt_merge.clocks import VectorClock
from crdt_merge.gossip import GossipState
import time
print('All imports OK')

In [ ]:
# Simulate Nord's 1.088B architecture (scaled down for demo)
def make_nord_state(seed, scale=1.0):
    rng = np.random.RandomState(seed)
    return {
        'sensory_zone.block_0.weight': rng.randn(512, 512).astype(np.float32) * scale,
        'sensory_zone.block_1.weight': rng.randn(512, 512).astype(np.float32) * scale,
        'association_zone.moe.expert_0': rng.randn(256, 256).astype(np.float32) * scale,
        'association_zone.moe.expert_1': rng.randn(256, 256).astype(np.float32) * scale,
        'memory_cortex.genesis.structural': rng.randn(96, 96).astype(np.float32) * scale,
        'memory_cortex.genesis.personal': rng.randn(96, 96).astype(np.float32) * scale,
        'memory_cortex.archive_grid': rng.randn(32, 128).astype(np.float32) * scale,
        'executive_zone.block_0.weight': rng.randn(512, 512).astype(np.float32) * scale,
        'temporal_spike_encoder.weight': rng.randn(512, 512).astype(np.float32) * scale,
        'lm_head.weight': rng.randn(512, 512).astype(np.float32) * scale,
        'stdp.weight_matrix': rng.randn(64, 64).astype(np.float32) * scale,
    }

# 3 people training on different hardware
state_laptop = make_nord_state(seed=1)   # RTX 5070, 27k steps
state_colab = make_nord_state(seed=2)    # Colab T4, 15k steps  
state_cloud = make_nord_state(seed=3)    # Cloud A100, 40k steps
print(f'Each checkpoint: {sum(v.nbytes for v in state_laptop.values())/1e6:.1f} MB')

In [ ]:
# FedAvg merge — sample-weighted averaging
fed = FederatedMerge(strategy='fedavg')
fed.submit('rtx5070_laptop', state_laptop, num_samples=30000)
fed.submit('colab_t4', state_colab, num_samples=50000)
fed.submit('cloud_a100', state_cloud, num_samples=120000)

result = fed.aggregate()
print(f'Merged {result.num_clients} nodes, {result.total_samples} total samples')
print(f'Contributions: {result.client_contributions}')
print(f'Layers merged: {len(result.model)}')

## 2. CRDT Checkpoint Merge — Order-independent (the key guarantee)

In [ ]:
# Prove merge(A,B) == merge(B,A) for ANY strategy
layer = 'sensory_zone.block_0.weight'

state_ab = CRDTMergeState('weight_average')
state_ab.add(state_laptop[layer], model_id='laptop', weight=0.4)
state_ab.add(state_cloud[layer], model_id='cloud', weight=0.6)

state_ba = CRDTMergeState('weight_average')
state_ba.add(state_cloud[layer], model_id='cloud', weight=0.6)
state_ba.add(state_laptop[layer], model_id='laptop', weight=0.4)

result_ab = np.asarray(state_ab.resolve())
result_ba = np.asarray(state_ba.resolve())

print(f'Commutativity check: max diff = {np.max(np.abs(result_ab - result_ba))}')
print(f'CRDT GUARANTEE HOLDS: {np.allclose(result_ab, result_ba)}')

## 3. Sparse Delta Sync — 93% sparsity = huge bandwidth savings

In [ ]:
# Simulate Nord's 93% sparsity during training
old_weights = np.zeros((1024, 1024), dtype=np.float32)
new_weights = old_weights.copy()

# Only 7% of neurons fire (change)
rng = np.random.RandomState(42)
mask = rng.random(old_weights.shape) < 0.07
new_weights[mask] = rng.randn(mask.sum()).astype(np.float32)

# Extract sparse delta
diff = new_weights.ravel() - old_weights.ravel()
nonzero = np.abs(diff) > 1e-7
indices = np.where(nonzero)[0]
values = diff[nonzero]

full_bytes = old_weights.nbytes
sparse_bytes = indices.nbytes + values.nbytes
sparsity = 1.0 - (len(indices) / old_weights.size)

print(f'Sparsity: {sparsity:.1%}')
print(f'Full tensor: {full_bytes/1024:.0f} KB')
print(f'Sparse delta: {sparse_bytes/1024:.0f} KB')
print(f'Compression: {full_bytes/sparse_bytes:.1f}x')
print(f'\nFor 1.088B params @ 93% sparsity:')
print(f'  Full sync: {1.088e9*4/1e9:.2f} GB')
print(f'  Sparse sync: {1.088e9*0.07*12/1e9:.2f} GB')

## 4. STDP as CRDT PNCounters

In [ ]:
# STDP potentiation = increment, depression = decrement
# Maps perfectly to PNCounter (positive-negative counter)

stdp_layer0 = PNCounter()

# Node A: 1234 potentiation events, 567 depression events
stdp_layer0.increment('rtx5070', 1234)   # synapses strengthened
stdp_layer0.decrement('rtx5070', 567)    # synapses weakened

# Node B: different training run
stdp_other = PNCounter()
stdp_other.increment('a100', 800)
stdp_other.decrement('a100', 100)

# CRDT merge — order independent
merged = stdp_layer0.merge(stdp_other)
print(f'Node A net STDP: {stdp_layer0.value}')
print(f'Node B net STDP: {stdp_other.value}')
print(f'Merged net STDP: {merged.value}')
print(f'Commutativity: {stdp_layer0.merge(stdp_other).value == stdp_other.merge(stdp_layer0).value}')

## 5. Training State Tracking — Loss, Sparsity, Emergence

In [ ]:
from crdt_merge.agentic import AgentState, SharedKnowledge

# Each training node as an agent
node_a = AgentState(agent_id='rtx5070_laptop')
node_a.add_fact('loss', 4.4, confidence=1.0)
node_a.add_fact('sparsity', 0.93, confidence=1.0)
node_a.add_fact('steps', 27000, confidence=1.0)
node_a.add_tag('emergence:cross_lingual_russian')
node_a.add_tag('emergence:memory_routing_shift')
node_a.increment('spikes_observed')

node_b = AgentState(agent_id='cloud_a100')
node_b.add_fact('loss', 4.1, confidence=1.0)
node_b.add_fact('sparsity', 0.94, confidence=1.0)
node_b.add_fact('steps', 50000, confidence=1.0)
node_b.add_tag('emergence:cross_lingual_russian')
node_b.add_tag('emergence:cross_lingual_chinese')

# Merge all observations
shared = SharedKnowledge.merge(node_a, node_b)
print('=== Merged Training State ===')
print(f'Facts: {list(shared.facts.keys())}')
for k, f in shared.facts.items():
    print(f'  {k}: {f.value} (confidence={f.confidence})')
print(f'Tags: {shared.tags}')

## 6. Genesis Memory State — CRDT-replicated persistent memory

In [ ]:
# Nord's Genesis Triple Memory replicated across nodes
# Structural bank (96 neurons), Personal bank (96), Auxiliary (64)
# Plus Archive Grid with 32 RAG slots

mem_a = LWWMap()  # Node A's memory state
mem_b = LWWMap()  # Node B's memory state
ts = time.time()

# Node A learns structural patterns
mem_a.set('structural_0', 0.95, timestamp=ts, node_id='node_a')
mem_a.set('structural_1', 0.82, timestamp=ts, node_id='node_a')
mem_a.set('archive_slot_0', 'subject-verb-object', timestamp=ts, node_id='node_a')

# Node B learns different patterns
mem_b.set('structural_0', 0.91, timestamp=ts+1, node_id='node_b')  # newer
mem_b.set('structural_2', 0.77, timestamp=ts, node_id='node_b')
mem_b.set('archive_slot_1', 'noun-phrase-structure', timestamp=ts, node_id='node_b')

# CRDT merge — LWW per key
merged_mem = mem_a.merge(mem_b)
print('Merged memory state:')
for k, v in sorted(merged_mem.value.items()):
    print(f'  {k}: {v}')
print(f'\nstructural_0 = {merged_mem.value["structural_0"]} (node_b wins — newer timestamp)')

## 7. Gossip Protocol — Peer-to-peer sync without central server

In [ ]:
# No parameter server needed. Each node gossips state to peers.
node1 = GossipState('laptop', fanout=2)
node2 = GossipState('colab', fanout=2)
node3 = GossipState('cloud', fanout=2)

# Each node records its training progress
node1.update('loss', 4.4)
node1.update('step', 27000)

node2.update('loss', 4.6)
node2.update('step', 15000)

node3.update('loss', 4.1)
node3.update('step', 50000)

# Anti-entropy: node1 compares digest with node2
digest1 = node1.digest()
digest2 = node2.digest()
push, pull = node1.anti_entropy_push_pull(digest2)
print(f'Node1 should push {len(push)} keys to Node2')
print(f'Node1 should pull {len(pull)} keys from Node2')

# Full merge
merged = node1.merge(node2).merge(node3)
print(f'\nMerged gossip state: {merged.size} live entries')
for key in ['loss', 'step']:
    val = merged.get(key)
    print(f'  {key}: {val}')

## 8. Vector Clocks — Causal ordering of training events

In [ ]:
from crdt_merge.clocks import Ordering

# Track causal ordering: which training events happened-before others
clock_a = VectorClock({'laptop': 27000, 'colab': 0})
clock_b = VectorClock({'laptop': 0, 'colab': 15000})
clock_c = VectorClock({'laptop': 5000, 'cloud': 50000})

print(f'A vs B: {clock_a.compare(clock_b)}')  # CONCURRENT
print(f'A vs C: {clock_a.compare(clock_c)}')  # CONCURRENT

merged_clock = clock_a.merge(clock_b).merge(clock_c)
print(f'\nMerged clock: {dict(merged_clock._clocks)}')
print('All events are now causally ordered.')

## Summary

| Nord Problem | crdt-merge Solution | CRDT Type |
|---|---|---|
| Budget exhaustion ($670) | Federated training across donated GPUs | FederatedMerge |
| 93% sparsity | Sparse delta sync (4-5x bandwidth savings) | GossipState + delta encoding |
| STDP weight updates | Potentiation/depression counters | PNCounter |
| Memory cortex (39% activation) | Replicated persistent memory | LWWMap |
| Cross-lingual emergence | Capability tracking (add-wins) | ORSet |
| Checkpoint merging | Order-independent merge | CRDTMergeState |
| No central server | Peer-to-peer gossip | GossipState |
| Causal ordering | Distributed clocks | VectorClock |